In [5]:
import pandas as pd
import ast

# Path to your CSV
input_file = "Campo_San_Lorenzo_with_AgeGroups.csv"
output_file = "Campo_San_Lorenzo_with_AgeGroups_cleanedd.csv"

# Load CSV
df = pd.read_csv(input_file)

# ---- 1. Replace empty cells in all columns with NA ----
df = df.fillna("NA")
df = df.replace(r'^\s*$', "NA", regex=True)


# ---- Helper function to clean age_groups column ----
def clean_age_group(value):
    if value == "NA":
        return "NA"

    # [] or [''] → NA
    if value.strip() in ["[]", "[']", "['']", "['']"]:
        return "NA"

    try:
        parsed = ast.literal_eval(value)

        if not parsed:  # empty list
            return "NA"

        # remove empty strings
        parsed = [x for x in parsed if str(x).strip() != ""]

        if not parsed:
            return "NA"

        # join items
        return ", ".join(parsed)

    except:
        # if value not list-like → return as is or NA if empty
        return value if value.strip() else "NA"


# ---- Apply cleaning to age_groups column ----
if "age_groups" in df.columns:
    df["age_groups"] = df["age_groups"].astype(str).apply(clean_age_group)



# ---- 2. Convert score columns from range (-1 to 1) to (0 to 2) ----

score_columns = ["TextBlob_Score", "Vader_Score", "CLIP_Visual_Score"]

for col in score_columns:
    if col in df.columns:
        # Convert numeric safely
        df[col] = pd.to_numeric(df[col], errors="coerce")

        # Shift range: -1 → 0 and 1 → 2
        df[col] = df[col] + 1

        # If NaN appears (from invalid inputs), replace with NA
        df[col] = df[col].fillna("NA")



# ---- Save final cleaned file ----
df.to_csv(output_file, index=False, mode='w')


print("✔ Cleaning complete!")
print(f"Saved cleaned file to: {output_file}")


✔ Cleaning complete!
Saved cleaned file to: Campo_San_Lorenzo_with_AgeGroups_cleanedd.csv


In [6]:
import pandas as pd

# Input cleaned CSV
input_file = "Campo_San_Lorenzo_with_AgeGroups_cleanedd.csv"

# Output CSV with selected columns
output_file = "Campo_San_Lorenzo_SelectedColumns.csv"

# Columns to keep
columns_to_export = [
    "id",
    "latitude",
    "longitude",
    "TextBlob_Score",
    "Vader_Score",
    "CLIP_Visual_Score"
]

# Load file
df = pd.read_csv(input_file)

# Select only the columns (ignore missing columns safely)
df_selected = df[columns_to_export]

# Save to new CSV
df_selected.to_csv(output_file, index=False)

print("✔ Export complete!")
print(f"Saved selected columns to: {output_file}")


✔ Export complete!
Saved selected columns to: Campo_San_Lorenzo_SelectedColumns.csv
